# FNO (masked-loss) vs. MUSIC — energy-density contours by η slice

Compares the TailMaskedLoss-retrained FNO model (`FNO4d` commit `dff91ba`,
see `PARTICLIZATION_INVESTIGATION.md`) against real MUSIC+iSS output,
overlaying CDF-based contours of the energy-density field at a chosen
spacetime-rapidity (η) bin, across 3 τ snapshots.

**"Truth" here is literally MUSIC's output** — these are the same ROOT
files (`AuAu_50ev_*_eta2_n33.root`) the model was trained/evaluated on,
generated by the real X-SCAPE/MUSIC/iSS pipeline, not a synthetic target.
This notebook runs the model fresh against held-out events from those
files — no live JetScape rerun needed.

All source settings live in **`eval_gcs_config_maskedloss.yaml`**. By
default it loads **only the 30-40% centrality file**: `train_torch.py`'s
train/verify split is a simple positional cut on the 4 training files
concatenated in centrality order, so for the full 200-event dataset the
held-out ~20% falls *entirely* inside 30-40%'s own block — the other three
centralities' events were all literally in the training set. To look at a
different centrality, edit `data_files` in the config to that file and set
`use_train_split: false` (its events are still real MUSIC output, just not
held-out — treat generalization claims from them accordingly), then re-run
from the top.

**Channels** (this model, n_features=5): `0=ε (energy density)`, `1=T`,
`2=vx`, `3=vy`, `4=vz`. Defaults to channel 0 below, the focus of this
notebook; `skinnycontour.plot_three_bins_contour`'s own per-channel title
labels (inherited from an older 4-channel model convention) don't match
this numbering past channel 0, so take the *dropdown* label as ground
truth if you switch channels, not the plot title.


In [ ]:
import io
import sys
import numpy as np
import matplotlib
matplotlib.use('Agg')          # non-interactive backend: no auto-display on figure
import matplotlib.pyplot as plt
plt.ioff()                     # belt-and-suspenders: keep interactive mode off
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output, Image

sys.path.insert(0, str(Path('.').resolve()))


In [ ]:
import eval_gcs_setup as _setup
try:
    _ctx = _setup.run_all(cfg_path='eval_gcs_config_maskedloss.yaml')
except Exception as _e:
    raise RuntimeError(f'Setup failed: {_e}') from _e

eval_cfg       = _ctx.eval_cfg
train_cfg      = _ctx.train_cfg
backend        = _ctx.backend
data_test      = _ctx.data_test
model_out_phys = _ctx.model_out_phys
y_true_phys    = _ctx.y_true_phys
eta_coords     = _ctx.eta_coords
tau_out_coords = _ctx.tau_out_coords
x_coords       = _ctx.x_coords
y_coords       = _ctx.y_coords
xy_extent      = _ctx.xy_extent
dtau           = _ctx.dtau
tau_min        = _ctx.tau_min
neta_eff       = _ctx.neta_eff
ntau_out       = _ctx.ntau_out

print(f'Loaded {data_test.shape[0]} event(s), {model_out_phys.shape[1]} channel(s), '
      f'{neta_eff} η bins, {ntau_out} τ steps.')


In [ ]:
_norm     = train_cfg.get('normalization', {})
norm_name = _norm.get('name', 'none') if isinstance(_norm, dict) else str(_norm)
_n_test   = data_test.shape[0]

_ws   = {'description_width': 'initial'}
_w400 = widgets.Layout(width='400px')

# --- channel labels for THIS model (n_features=5: ε, T, vx, vy, vz) --------
# Not the same numbering skinnycontour.plot_three_bins_contour's own title
# text assumes (an older 4-channel ε/vy/vx/vz convention) -- the dropdown
# label below is the correct one; the plot's title text is only right for
# channel 0.
_channel_options = [
    '0 — ε (energy density)',
    '1 — T (temperature)',
    '2 — vx',
    '3 — vy',
    '4 — vz',
][:model_out_phys.shape[1]]

cont_event_ui = widgets.IntSlider(
    min=0, max=_n_test - 1, value=0, step=1,
    description='Event:', layout=_w400, style=_ws,
)
cont_ieta_ui = widgets.IntSlider(
    min=0, max=neta_eff - 1, value=int(np.argmin(np.abs(eta_coords - 0.0))), step=1,
    description='η bin:', layout=_w400, style=_ws,
)
cont_channel_ui = widgets.Dropdown(
    options=_channel_options,
    value=_channel_options[0], description='Feature / channel:', style=_ws,
)
contour_iT_ui = widgets.Text(
    value='40,-1',
    description='',
    layout=widgets.Layout(width='120px'),
)
cdfrats_ui = widgets.Text(
    value='0.05,0.1,0.2,0.3,0.4,0.6',
    description='',
    layout=widgets.Layout(width='260px'),
)
# model_out_phys/y_true_phys are already denormalized to physical units
# (GeV/fm^3) by eval_gcs_setup's inference step -- rescale_energy=True
# would apply ANOTHER (e/tau)^(4/3) transform on top of that, which is only
# correct if the arrays were still in normalized (train-space) units. Off
# by default here on purpose, unlike eval_gcs_clean.ipynb's default (which
# assumed a different data flow). Leave off unless you know you want it.
rescale_energy_ui = widgets.Checkbox(
    value=False,
    description='Rescale energy: (e/τ)^(4/3)  [NOT needed for physical-unit data -- see comment above]',
    style=_ws,
)


In [ ]:
_controls = widgets.VBox([
    widgets.HBox([cont_event_ui, cont_ieta_ui, cont_channel_ui]),
    widgets.HBox([
        widgets.VBox([
            widgets.Label('2nd and 3rd τ-step indices (comma-sep; 1st panel always 0; -1=last non-zero):'),
            contour_iT_ui,
        ]),
        widgets.VBox([
            widgets.Label('CDF ratios for contour levels (comma-sep):'),
            cdfrats_ui,
        ]),
    ]),
    widgets.HBox([
        widgets.HTML(f'<b>Normalizer:</b> <code>{norm_name}</code>&nbsp;&nbsp;'),
        rescale_energy_ui,
    ]),
])

_run_btn = widgets.Button(
    description='Plot contours',
    button_style='primary',
    icon='play',
    layout=widgets.Layout(width='200px', margin='8px 0'),
)
_plot_out = widgets.Output()


def _show(fig):
    """Render fig to a PNG buffer and display as Image, then close the figure
    (matplotlib's Agg backend never auto-displays; this is the one render path)."""
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=fig.dpi, bbox_inches='tight')
    plt.close(fig)
    buf.seek(0)
    display(Image(data=buf.read()))


def _run_eval(b):
    from skinnycontour import plot_three_bins_contour

    with _plot_out:
        clear_output(wait=True)

        _event          = int(cont_event_ui.value)
        _channel        = int(cont_channel_ui.value.split()[0])
        _rescale_energy = bool(rescale_energy_ui.value)
        _raw_iT = [t.strip() for t in contour_iT_ui.value.split(',') if t.strip()]
        if len(_raw_iT) < 2:
            _raw_iT = (_raw_iT + ['-1', '-1'])[:2]
        elif len(_raw_iT) > 2:
            _raw_iT = _raw_iT[:2]
        _contour_iT = (0,) + tuple(int(t) for t in _raw_iT)
        _cdfrats    = [float(x.strip()) for x in cdfrats_ui.value.split(',') if x.strip()]
        _tau0       = tau_min + dtau
        _ieta       = int(cont_ieta_ui.value)
        _eta_phys   = eta_coords[_ieta]

        _fig, _axes = plt.subplots(1, 3, figsize=(16.0, 4.5))
        plot_three_bins_contour(
            model_out_phys, y_true_phys,
            iparam=_channel, event=_event,
            tau0=_tau0, taudelta=dtau, iT=_contour_iT,
            cdfrats=_cdfrats, ieta=_ieta, tag=f'η={_eta_phys:.2f}',
            rescale_energy=_rescale_energy,
            x_coords=x_coords, y_coords=y_coords,
            fig=_fig, axes=_axes,
        )
        _show(_fig)

        # Relative L2 error vs tau, all eta bins, for this channel -- quick
        # context on which eta/tau region the chosen snapshot sits in.
        _denom  = np.mean(y_true_phys[:, _channel] ** 2, axis=(0, 1, 2))
        _num    = np.mean((model_out_phys[:, _channel] - y_true_phys[:, _channel]) ** 2, axis=(0, 1, 2))
        _rel_l2 = np.sqrt(_num / np.maximum(_denom, 1e-30))
        _fig2, _ax2 = plt.subplots(figsize=(8, 4))
        for _ie in range(neta_eff):
            _ax2.plot(tau_out_coords, _rel_l2[_ie], label=f'η={eta_coords[_ie]:.2f}')
        _ax2.axvline(_tau0 + _contour_iT[0] * dtau, color='gray', ls=':', lw=1)
        _ax2.set_xlabel('τ (fm/c)')
        _ax2.set_ylabel('Relative L² error')
        _ax2.set_title(f'Relative L² error vs τ (all η bins, test set) — channel {_channel}')
        _ax2.legend(fontsize=7, ncol=2)
        _ax2.grid(True, alpha=0.3)
        _ax2.set_ylim(0, 2)
        _fig2.tight_layout()
        _show(_fig2)


_run_btn.on_click(_run_eval)

display(widgets.VBox([_controls, _run_btn, _plot_out]))


---

## Azimuthally-averaged radial energy-density profile

The CDF-contour plots above show percentile *boundaries*, not the actual
shape of the energy-density profile -- they can't directly answer "at what
radius does FNO's energy density fall short of MUSIC's?" This cell builds
that directly: bin every (x,y) cell in the chosen η slice by its transverse
radius `r = sqrt(x²+y²)`, average the energy density within each radial bin
(and across **all held-out events**, not just one, for a cleaner signal
than the single-event contour panels above), and plot FNO vs. truth plus
their ratio vs. r.

Motivation: peak (r≈0) energy density and the energy-weighted RMS
transverse radius both match MUSIC closely (see
`PARTICLIZATION_INVESTIGATION.md`'s "Peak energy density" and
"Transverse-footprint hypothesis" sections), yet the *total* (x,y-summed)
energy density is uniformly ~20-40% low across the whole η range. Matching
peak + matching weighted spread + low total integral is inconsistent with
a deficit concentrated at the outer edge specifically (that would pull the
weighted radius *inward*, which isn't observed, and the thin low-density
edge doesn't hold enough of the total energy budget to explain a
20-40% shortfall by itself) -- it points instead at a deficit somewhere in
the bulk/shoulder of the profile. This plot checks that directly instead of
inferring it from moments.


In [ ]:
def radial_profile(data_xy, x_coords, y_coords, r_edges):
    """data_xy: [N_events, nx, ny] physical-unit field (one channel, one
    eta bin, one tau step, already selected by the caller). Returns
    (r_centers, mean_profile) -- mean over every event AND every cell
    falling in each radial bin (not energy-weighted; a plain spatial mean,
    since the question here is 'how much energy sits at this radius',
    not 'where does the energy-weighted centroid sit')."""
    xx, yy = np.meshgrid(x_coords, y_coords, indexing='ij')
    rr = np.sqrt(xx**2 + yy**2)
    r_centers = 0.5 * (r_edges[:-1] + r_edges[1:])
    profile = np.full(len(r_centers), np.nan)
    for i in range(len(r_centers)):
        mask = (rr >= r_edges[i]) & (rr < r_edges[i + 1])
        if mask.sum() == 0:
            continue
        profile[i] = data_xy[:, mask].mean()
    return r_centers, profile


_w350 = widgets.Layout(width='350px')

radial_ieta_ui = widgets.IntSlider(
    min=0, max=neta_eff - 1, value=int(np.argmin(np.abs(eta_coords - 0.0))), step=1,
    description='η bin:', layout=_w400, style=_ws,
)
radial_channel_ui = widgets.Dropdown(
    options=_channel_options, value=_channel_options[0],
    description='Feature / channel:', style=_ws,
)
radial_iT_ui = widgets.Text(
    value='0,40,-1',
    description='',
    layout=_w350,
)
radial_dr_ui = widgets.BoundedFloatText(
    value=0.75, min=0.1, max=5.0, step=0.05,
    description='Radial bin width (fm):', style=_ws,
)
radial_rmax_ui = widgets.BoundedFloatText(
    value=10.0, min=1.0, max=14.0, step=0.5,
    description='Max radius (fm):', style=_ws,
)
# Ratio = model/truth blows up to nonsense (and once truth has frozen out
# for most held-out events at a given eta/tau -- e.g. 30-40% centrality by
# tau~8 fm -- can read 10x+) wherever truth is near its own padding floor.
# Mask the ratio panel (not the raw profile panel -- that's fine on a log
# scale) wherever truth is below this fraction of that SAME profile's own
# peak, so it adapts to whatever channel/tau/eta is selected rather than a
# fixed absolute GeV/fm^3 cutoff.
radial_thresh_ui = widgets.BoundedFloatText(
    value=0.01, min=0.0, max=0.5, step=0.005,
    description='Ratio-panel mask: truth < this × peak → excluded:', style=_ws,
    layout=widgets.Layout(width='400px'),
)

radial_controls = widgets.VBox([
    widgets.HBox([radial_ieta_ui, radial_channel_ui]),
    widgets.HBox([
        widgets.VBox([widgets.Label('τ-step indices (comma-sep, -1=last non-zero):'), radial_iT_ui]),
        radial_dr_ui,
        radial_rmax_ui,
    ]),
    widgets.HBox([radial_thresh_ui]),
])

radial_run_btn = widgets.Button(
    description='Plot radial profile', button_style='primary', icon='play',
    layout=widgets.Layout(width='220px', margin='8px 0'),
)
radial_out = widgets.Output()


def _run_radial(b):
    with radial_out:
        clear_output(wait=True)

        _ieta    = int(radial_ieta_ui.value)
        _channel = int(radial_channel_ui.value.split()[0])
        _eta_phys = eta_coords[_ieta]
        _dr   = float(radial_dr_ui.value)
        _rmax = float(radial_rmax_ui.value)
        _thresh_frac = float(radial_thresh_ui.value)
        _r_edges = np.arange(0.0, _rmax + _dr, _dr)

        _raw_iT = [t.strip() for t in radial_iT_ui.value.split(',') if t.strip()]
        _iTs = []
        for t in _raw_iT:
            t = int(t)
            if t == -1:
                t = ntau_out - 1
                while np.all(np.abs(y_true_phys[:, 0, :, :, _ieta, t]) < 1e-5):
                    t -= 1
                    if t == 0:
                        break
            _iTs.append(t)

        _fig, _axes = plt.subplots(2, len(_iTs), figsize=(5.0 * len(_iTs), 7.5),
                                   squeeze=False, sharex='col')
        _fig.suptitle(f'Radial energy-density profile, η={_eta_phys:.2f}, '
                      f'averaged over {y_true_phys.shape[0]} held-out event(s)',
                      fontsize=13, y=1.02)

        for _col, _iT in enumerate(_iTs):
            _tau_phys = tau_out_coords[_iT]
            _truth_xy = y_true_phys[:, _channel, :, :, _ieta, _iT]
            _model_xy = model_out_phys[:, _channel, :, :, _ieta, _iT]
            _r, _p_truth = radial_profile(_truth_xy, x_coords, y_coords, _r_edges)
            _, _p_model = radial_profile(_model_xy, x_coords, y_coords, _r_edges)

            _ax_top = _axes[0, _col]
            _ax_top.plot(_r, _p_truth, color='#2a78d6', lw=2.2, marker='o', ms=4, label='MUSIC (truth)')
            _ax_top.plot(_r, _p_model, color='#eb6834', lw=2.2, marker='o', ms=4, label='FNO')
            if _channel == 0:
                _ax_top.set_yscale('log')
            _ax_top.set_title(f'τ = {_tau_phys:.2f} fm/c', fontsize=11)
            _ax_top.grid(True, alpha=0.3)
            if _col == 0:
                _ax_top.set_ylabel('mean value in radial bin')
                _ax_top.legend(fontsize=9)

            _ax_bot = _axes[1, _col]
            _peak = np.nanmax(np.abs(_p_truth))
            _valid = np.abs(_p_truth) >= _thresh_frac * _peak
            with np.errstate(divide='ignore', invalid='ignore'):
                _ratio = np.where(_valid, _p_model / _p_truth, np.nan)
            _ax_bot.plot(_r, _ratio, color='#6250d6', lw=2.0, marker='o', ms=4)
            if (~_valid).any():
                _ax_bot.plot(_r[~_valid], np.ones(np.sum(~_valid)), 'x', color='lightgray',
                            ms=8, label=f'truth < {_thresh_frac:.1%} of peak (excluded)')
                _ax_bot.legend(fontsize=7, loc='upper left')
            _ax_bot.axhline(1.0, color='gray', ls='--', lw=1)
            _ax_bot.set_ylim(0, 2)
            _ax_bot.set_xlabel('r (fm)')
            _ax_bot.grid(True, alpha=0.3)
            if _col == 0:
                _ax_bot.set_ylabel('FNO / MUSIC ratio')

        _fig.tight_layout()
        _show(_fig)


radial_run_btn.on_click(_run_radial)
display(widgets.VBox([radial_controls, radial_run_btn, radial_out]))
